# Generalised ISP — Multi-Model Clinical Summarisation
Iterative Self-improving Prompt (ISP) loop, generalised across LLMs.

**To switch models, change only `SELECTED_MODEL` in Cell 4.**
Supported: `qwen2.5`, `ministral_8b`, `llama3.1_8b`, `qwen3`.

Changes vs. original:
- Generated summary length **fixed to 150 tokens** (not 70% of reference).
- Same loaded LLM is used for **grammar scoring** (with LanguageTool fallback).
- Reports **avg BERT-F1, avg ROUGE-L, summarisation score (0.6·ROUGE-L + 0.4·BERT-F1), avg grammar**.
- Identifies and saves the **best-performing prompt**.

In [ ]:
# ── CELL 1 — Install dependencies ────────────────────────────────────────────
import subprocess
subprocess.run([
    "pip", "install", "-q",
    "transformers", "accelerate", "bitsandbytes",
    "datasets", "evaluate", "rouge-score",
    "bert-score", "pandas", "matplotlib",
    "language_tool_python==2.9.2", "nltk", "sentencepiece",
], check=True)
subprocess.run(["pip", "install", "-q", "-U", "transformers"], check=True)
print("✅ Dependencies installed")

In [ ]:
# ── CELL 2 — Imports ─────────────────────────────────────────────────────────
import os
import re
import gc
import json
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)
import evaluate as hf_evaluate
from bert_score import score as bert_score_fn
import transformers

# Grammar-scoring deps
import nltk
import language_tool_python
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

random.seed(42)
torch.random.manual_seed(0)

print("✅ All imports successful")
print(f"   transformers : {transformers.__version__}")
print(f"   torch        : {torch.__version__}")
print(f"   CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU          : {torch.cuda.get_device_name(0)}")
    print(f"   GPU memory   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("   ⚠️  No GPU detected")

In [ ]:
# ── CELL 3 — HuggingFace token (for gated models) ────────────────────────────
# Llama-3.1 and Ministral are GATED → require an HF access token with access granted.
# Provide your token in ANY of these ways (checked in order):
#   1. Kaggle "Secrets"  → add a secret named  HF_TOKEN
#   2. Environment var   → os.environ["HF_TOKEN"]
#   3. Paste below       → HF_TOKEN_FALLBACK = "hf_xxx"
HF_TOKEN_FALLBACK = ""   # ← optionally paste your token here

HF_TOKEN = None
# 1. Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ HF token loaded from Kaggle Secrets")
except Exception:
    pass
# 2. Environment variable
if not HF_TOKEN:
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
    if HF_TOKEN:
        print("✅ HF token loaded from environment variable")
# 3. Fallback paste
if not HF_TOKEN and HF_TOKEN_FALLBACK:
    HF_TOKEN = HF_TOKEN_FALLBACK
    print("✅ HF token loaded from fallback variable")

if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("✅ Logged in to HuggingFace Hub")
    except Exception as e:
        print(f"   ⚠️  hub login skipped: {e}")
else:
    print("⚠️  No HF token found. Open models (Qwen) work; gated models (Llama/Ministral) will fail.")

In [ ]:
# ── CELL 4 — Model registry & configuration ──────────────────────────────────
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CHANGE ONLY THIS LINE TO SWITCH MODELS                                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
SELECTED_MODEL = "qwen2.5"      # one of: qwen2.5 | ministral_8b | llama3.1_8b | qwen3

# Registry: friendly name → HF repo id + flags.
#   gated   : needs HF token with granted access
#   dtype   : compute dtype for 4-bit quant
MODEL_REGISTRY = {
    "qwen2.5":      {"repo": "Qwen/Qwen2.5-7B-Instruct",            "gated": False, "dtype": torch.float16,  "display": "Qwen2.5-7B"},
    "ministral_8b": {"repo": "mistralai/Ministral-8B-Instruct-2410", "gated": True,  "dtype": torch.bfloat16, "display": "Ministral-8B"},
    "llama3.1_8b":  {"repo": "meta-llama/Llama-3.1-8B-Instruct",     "gated": True,  "dtype": torch.bfloat16, "display": "Llama-3.1-8B"},
    "qwen3":        {"repo": "Qwen/Qwen3-8B",                        "gated": False, "dtype": torch.bfloat16, "display": "Qwen3-8B"},
}

assert SELECTED_MODEL in MODEL_REGISTRY, f"Unknown model '{SELECTED_MODEL}'. Choose from {list(MODEL_REGISTRY)}"
_cfg          = MODEL_REGISTRY[SELECTED_MODEL]
MODEL_NAME    = _cfg["repo"]
MODEL_GATED   = _cfg["gated"]
MODEL_TAG     = SELECTED_MODEL                       # used in filenames
MODEL_DISPLAY = _cfg["display"]                      # used in graph titles

# ── Compute dtype: auto-pick based on GPU ─────────────────────────────────────
# bfloat16 is SLOW on GPUs without native bf16 (e.g. Tesla T4, common on Kaggle
# free tier) — there it runs many times slower than fp16. On Ampere/Hopper
# (A100, H100, RTX 30/40) bf16 runs at full speed and is numerically safer.
# FORCE_FP16=True overrides everything and uses fp16 (fastest on old GPUs).
FORCE_FP16 = True   # ← set False to let bf16 be used where the GPU supports it

if FORCE_FP16:
    MODEL_DTYPE = torch.float16
    _dtype_reason = "FORCE_FP16=True"
elif torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    MODEL_DTYPE = _cfg["dtype"]
    _dtype_reason = "GPU supports bf16 — using registry default"
else:
    MODEL_DTYPE = torch.float16
    _dtype_reason = "GPU lacks fast bf16 — falling back to fp16"

if MODEL_GATED and not HF_TOKEN:
    raise RuntimeError(
        f"'{SELECTED_MODEL}' is gated and needs an HF token with access. "
        f"Set HF_TOKEN (see Cell 3) and request access to {MODEL_NAME} on HuggingFace."
    )

# ── Other model / NLI config ─────────────────────────────────────────────────
NLI_MODEL_NAME      = "cross-encoder/nli-deberta-v3-large"   # textual entailment
MAX_INPUT_TOKENS    = 3072
BERT_MODEL          = "roberta-large"
CHECKPOINT_EVERY    = 10

# ── ISP parameters ────────────────────────────────────────────────────────────
ITERATIONS          = 3
SEARCH_CASES        = 592

# ── Token length parameters  (FIXED 150 TOKENS) ───────────────────────────────
FIXED_TOKENS        = 150       # ← generated summary length fixed to exactly this
WORD_TOKEN_RATIO    = 0.75
REPETITION_PENALTY  = 1.2

# ── Prompt improvement parameters ─────────────────────────────────────────────
MAX_CRITIQUE_TOKENS = 200
MAX_PATCH_TOKENS    = 400
PLACEHOLDER_WORD    = "{target_words}"
DISPLAY_WORD_COUNT  = 150

# Word-count instruction switch.
#   False → NO word-count text in the prompt; length controlled ONLY by the
#           150-token cap. Placeholder logic is fully bypassed (and the
#           critique/patch loop will NOT re-insert any word target).
#   True  → keep the dynamic '{target_words}' placeholder behaviour.
USE_WORD_COUNT      = False

# ── Summarisation score weights ───────────────────────────────────────────────
W_ROUGEL            = 0.6
W_BERT              = 0.4

# ── Final-run behaviour ───────────────────────────────────────────────────────
# If True AND the best prompt was already scored on the FULL dataset during the
# ISP iterations (i.e. search set == full set), the 4th "final" pass is skipped
# and those existing rows are reused — saving ~1/4 of total compute.
# If the search set is a subset of the full dataset, a fresh final run still
# happens regardless of this flag.
SKIP_REDUNDANT_FINAL = True

print("✅ Configuration ready")
print(f"   SELECTED_MODEL   : {SELECTED_MODEL}")
print(f"   MODEL_NAME       : {MODEL_NAME}  (gated={MODEL_GATED})")
print(f"   ITERATIONS       : {ITERATIONS}")
print(f"   SEARCH_CASES     : {SEARCH_CASES}")
print(f"   FIXED_TOKENS     : {FIXED_TOKENS} (max_new = min_new = {FIXED_TOKENS})")
print(f"   Summ. score      : {W_ROUGEL}·ROUGE-L + {W_BERT}·BERT-F1")
print(f"   Compute dtype    : {MODEL_DTYPE}  ({_dtype_reason})")
if torch.cuda.is_available():
    print(f"   GPU              : {torch.cuda.get_device_name(0)}  | native bf16: {torch.cuda.is_bf16_supported()}")

In [ ]:
# ── CELL 5 — Output paths & checkpoint check ─────────────────────────────────
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def _p(name):  # model-tagged path helper
    return os.path.join(OUTPUT_DIR, f"{MODEL_TAG}_{name}")

ITER_RESULTS_CSV = _p("isp_iteration_results.csv")
PROMPT_HIST_JSON = _p("isp_prompt_history.json")
FINAL_CSV        = _p("isp_final_592.csv")
METRICS_JSON     = _p("isp_metrics_summary.json")
CHECKPOINT_FILE  = _p("isp_checkpoint.json")
GRAPH_HIGH_ROUGE = _p("top10_high_rougeL.png")
GRAPH_LOW_ROUGE  = _p("top10_low_rougeL.png")
GRAPH_HIGH_BERT  = _p("top10_high_bert_f1.png")
GRAPH_LOW_BERT   = _p("top10_low_bert_f1.png")
GRAPH_ROUGE_COMP = _p("isp_iter_rouge_comparison.png")
GRAPH_BERT_COMP  = _p("isp_iter_bert_comparison.png")
GRAPH_SCORE_TBL  = _p("score_comparison.png")

print(f"✅ Output directory : {OUTPUT_DIR}")
print(f"   File prefix      : {MODEL_TAG}_*")

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        ckpt_peek = json.load(f)
    print(f"\n⚠️  CHECKPOINT DETECTED for {MODEL_TAG}")
    print(f"   Completed iterations : {ckpt_peek.get('completed_iterations', [])}")
    print(f"   Current iteration    : {ckpt_peek.get('current_iteration', 1)}")
else:
    print(f"\n   No checkpoint — starting fresh")

In [ ]:
# ── CELL 6 — Dataset paths ────────────────────────────────────────────────────
# Point this to YOUR Kaggle dataset. Expected structure (same as original):
#   <BASE>/Data/Short/fulltext/*.txt
#   <BASE>/Data/Short/summaries/*_sum.txt
KAGGLE_DATASET_BASE = "/kaggle/input/datasets/anshumanmoharana07/biolaysumm"

SHORT_FULL_PATH = os.path.join(KAGGLE_DATASET_BASE, "Data", "Short", "fulltext")
SHORT_SUM_PATH  = os.path.join(KAGGLE_DATASET_BASE, "Data", "Short", "summaries")

print("Path check:")
for label, p in [("Short fulltext", SHORT_FULL_PATH), ("Short summaries", SHORT_SUM_PATH)]:
    if os.path.exists(p):
        n = len([f for f in os.listdir(p) if f.endswith(".txt")])
        print(f"  ✅  {label:<18} : {p}  ({n} files)")
    else:
        print(f"  ❌  NOT FOUND : {p}")

In [ ]:
# ── CELL 7 — Load dataset ────────────────────────────────────────────────────
def load_short_dataset(full_path, summary_path):
    all_ft  = sorted([f for f in os.listdir(full_path) if f.endswith(".txt")])
    sum_set = set(os.listdir(summary_path))
    cases, summaries, filenames, skipped = [], [], [], []

    for ft_file in all_ft:
        stem     = ft_file[:-4]
        sum_file = stem + "_sum.txt"
        if sum_file not in sum_set:
            skipped.append(ft_file); continue
        with open(os.path.join(full_path, ft_file), "r", encoding="utf-8", errors="ignore") as f:
            case_text = f.read().strip()
        with open(os.path.join(summary_path, sum_file), "r", encoding="utf-8", errors="ignore") as f:
            sum_text = f.read().strip()
        if not case_text or not sum_text:
            skipped.append(ft_file); continue
        cases.append(case_text)
        summaries.append(sum_text)
        filenames.append(stem)

    print(f"✅ Loaded  : {len(cases)} cases")
    print(f"   Skipped : {len(skipped)}")
    return Dataset.from_dict({"filename": filenames, "case": cases, "summary": summaries})


print("Loading Short dataset ...")
short_dataset = load_short_dataset(SHORT_FULL_PATH, SHORT_SUM_PATH)
print(f"\nTotal cases : {len(short_dataset)}")

SEARCH_CASES   = min(SEARCH_CASES, len(short_dataset))
search_indices = list(range(SEARCH_CASES))
search_dataset = short_dataset.select(search_indices)
print(f"Search subset : {len(search_dataset)} cases (indices 0–{SEARCH_CASES-1})")

In [ ]:
# ── CELL 8 — Load generation model (generalised) ─────────────────────────────
print("Clearing GPU memory ...")
for _name in ["model", "tokenizer"]:
    try:
        del globals()[_name]
    except (KeyError, NameError):
        pass
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    free_mem = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
    print(f"  GPU free : {free_mem:.2f} GB\n")

print(f"Loading: {MODEL_NAME}  (tag={MODEL_TAG}) ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=MODEL_DTYPE,
    bnb_4bit_use_double_quant=True,
)

_auth = {"token": HF_TOKEN} if HF_TOKEN else {}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, **_auth)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Tokenizer ready")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    **_auth,
)
model.eval()
print(f"✅ Model loaded on : {next(model.parameters()).device}")
print(f"   GPU memory used : {torch.cuda.memory_allocated()/1e9:.2f} GB")

# Some models (e.g. Qwen3) support a "thinking" mode in the chat template.
# We disable it for clean, deterministic summaries when the kwarg is supported.
SUPPORTS_ENABLE_THINKING = "enable_thinking" in getattr(
    tokenizer.apply_chat_template, "__doc__", "") or "qwen3" in MODEL_TAG

def build_chat_text(messages):
    """Apply the model's chat template, disabling thinking mode if available."""
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
        )

print(f"   enable_thinking disabled where supported")

In [ ]:
# ── CELL 9 — Load NLI model (textual entailment) ─────────────────────────────
print(f"Loading NLI model: {NLI_MODEL_NAME} ...")

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_NAME)
nli_model     = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_NAME)
nli_model.eval()
nli_model.to("cuda" if torch.cuda.is_available() else "cpu")

print(f"   NLI label map : {nli_model.config.id2label}")
print(f"✅ NLI model ready")
print(f"   GPU memory used : {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── CELL 10 — Textual entailment helpers ─────────────────────────────────────
NLI_MAX_LEN = 512

_id2label = nli_model.config.id2label
_label2id = {v.lower(): k for k, v in _id2label.items()}

ENT_IDX  = _label2id.get("entailment",    _label2id.get("entail", 1))
NEU_IDX  = _label2id.get("neutral",       2)
CON_IDX  = _label2id.get("contradiction", _label2id.get("contradict", 0))

print(f"NLI index mapping → entailment:{ENT_IDX}  neutral:{NEU_IDX}  contradiction:{CON_IDX}")


def run_nli(premise: str, hypothesis: str) -> dict:
    inputs = nli_tokenizer(
        premise, hypothesis,
        return_tensors="pt", truncation=True,
        max_length=NLI_MAX_LEN, padding=True,
    ).to(nli_model.device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0].cpu().tolist()
    ent  = round(probs[ENT_IDX], 4)
    neu  = round(probs[NEU_IDX], 4)
    con  = round(probs[CON_IDX], 4)
    entailed = bool(ent > neu and ent > con)
    return {"entailment_prob": ent, "neutral_prob": neu,
            "contradiction_prob": con, "entailed": entailed}


def compute_entailment(document, reference_summary, generated_summary) -> dict:
    ref_result = run_nli(document, reference_summary)
    gen_result = run_nli(document, generated_summary)
    both = bool(ref_result["entailed"] and gen_result["entailed"])
    return {
        "entailment_prob_reference":    ref_result["entailment_prob"],
        "neutral_prob_reference":        ref_result["neutral_prob"],
        "contradiction_prob_reference":  ref_result["contradiction_prob"],
        "reference_entailed":            ref_result["entailed"],
        "entailment_prob_generated":     gen_result["entailment_prob"],
        "neutral_prob_generated":        gen_result["neutral_prob"],
        "contradiction_prob_generated":  gen_result["contradiction_prob"],
        "generated_entailed":            gen_result["entailed"],
        "both_entailed":                 both,
    }


print("✅ Textual entailment helpers ready")

In [ ]:
# ── CELL 11 — Sentence controls ──────────────────────────────────────────────
class SentenceEndStoppingCriteria(StoppingCriteria):
    """Stop at sentence boundary (. ! ?) after min_tokens generated."""
    def __init__(self, tokenizer, min_tokens, input_len):
        self.min_tokens = min_tokens
        self.input_len  = input_len
        self.sentence_end_ids = set()
        for punct in [".", "!", "?", ". ", "! ", "? ", ".\n", "...", '."', '!"']:
            ids = tokenizer.encode(punct, add_special_tokens=False)
            if ids:
                self.sentence_end_ids.add(ids[-1])
        if tokenizer.eos_token_id is not None:
            self.sentence_end_ids.add(tokenizer.eos_token_id)

    def __call__(self, input_ids, scores, **kwargs):
        new_tokens = input_ids.shape[1] - self.input_len
        if new_tokens < self.min_tokens:
            return False
        return input_ids[0, -1].item() in self.sentence_end_ids


def trim_to_last_fullstop(text: str) -> str:
    text = text.strip()
    if not text:
        return text
    if text[-1] in ('.', '!', '?'):
        return text
    last_idx = max(text.rfind('.'), text.rfind('!'), text.rfind('?'))
    if last_idx > 0:
        return text[:last_idx + 1].strip()
    return text


def remove_incomplete_sentence(text: str) -> str:
    if not text or not text.strip():
        return text
    text = text.strip()
    text = trim_to_last_fullstop(text)
    if text.endswith(('.', '!', '?', '."', '!"', '?"')):
        return text
    sentences = re.split(r'(?<=[.!?])\s+', text)
    if len(sentences) <= 1:
        return text if len(text.split()) >= 10 else ""
    complete = [s for s in sentences[:-1] if s.strip().endswith(('.', '!', '?'))]
    return ' '.join(complete).strip() if complete else text


print("✅ Sentence controls ready")

In [ ]:
# ── CELL 12 — Token length helpers  (FIXED 150 TOKENS) ───────────────────────
def count_tokens(text):
    return len(tokenizer.encode(text, add_special_tokens=False))


def get_fixed_length(reference_text):
    """Length is now FIXED, independent of the reference length.
       max_new = min_new = FIXED_TOKENS.  ref_len still returned for logging."""
    ref_len = count_tokens(reference_text)
    max_new = FIXED_TOKENS
    min_new = FIXED_TOKENS
    tw      = max(20, int(FIXED_TOKENS * WORD_TOKEN_RATIO))
    return max_new, min_new, tw, ref_len


def truncate_to_budget(text, budget):
    ids = tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= budget:
        return text
    return tokenizer.decode(ids[:budget], skip_special_tokens=True) + "\n[...truncated...]"


print(f"✅ Token length helpers ready (FIXED {FIXED_TOKENS} tokens)")

In [ ]:
# ── CELL 13 — Generation helpers ─────────────────────────────────────────────
def _single_generate(prompt_template, case_text, reference_text):
    """One generation pass — fixed 150-token target."""
    max_new, min_new, target_words, ref_len = get_fixed_length(reference_text)

    prompt     = prompt_template.replace(PLACEHOLDER_WORD, str(target_words)) if USE_WORD_COUNT else prompt_template
    budget     = max(MAX_INPUT_TOKENS - 200, 256)
    case_trunc = truncate_to_budget(case_text, budget)

    messages = [
        {"role": "system", "content": "You are a clinical summarization expert."},
        {"role": "user",   "content": f"{prompt}\n\nClinical Case Report:\n{case_trunc}"}
    ]
    text = build_chat_text(messages)

    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=MAX_INPUT_TOKENS, padding=False,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    stopping_criteria = StoppingCriteriaList([
        SentenceEndStoppingCriteria(tokenizer, min_new, input_len)
    ])

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens     = max_new,
            min_new_tokens     = min_new,
            do_sample          = False,
            repetition_penalty = REPETITION_PENALTY,
            stopping_criteria  = stopping_criteria,
            eos_token_id       = tokenizer.eos_token_id,
            pad_token_id       = tokenizer.pad_token_id,
        )

    new_ids     = output_ids[0][input_len:]
    raw_summary = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    trimmed = trim_to_last_fullstop(raw_summary)
    clean   = remove_incomplete_sentence(trimmed)
    if not clean.strip():
        clean = raw_summary if raw_summary else "No summary generated."
    return clean, max_new, min_new, target_words, ref_len


def generate_summary(prompt_template, case_text, reference_text):
    """Single-pass generation — no retries. The model already produces at least
    min_new_tokens; we accept the first generation and trim to the last full stop."""
    clean, max_new, min_new, target_words, ref_len = _single_generate(
        prompt_template, case_text, reference_text)
    clean = trim_to_last_fullstop(clean)
    return clean, max_new, min_new, target_words, ref_len


def generate_model_response(instruction, max_tokens, temperature=0.7, top_p=0.9):
    """Generate critique/patch/grammar response — no sentence stopping criteria."""
    messages = [
        {"role": "system", "content": "You are an expert NLP prompt engineer."},
        {"role": "user",   "content": instruction}
    ]
    text = build_chat_text(messages)
    inputs = tokenizer(
        text, return_tensors="pt",
        truncation=True, max_length=2048, padding=False,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            do_sample          = True,
            temperature        = temperature,
            top_p              = top_p,
            repetition_penalty = 1.15,
            eos_token_id       = tokenizer.eos_token_id,
            pad_token_id       = tokenizer.pad_token_id,
        )
    new_ids = output_ids[0][input_len:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


print("✅ Generation helpers ready (fixed-150, single-pass + full-stop trimming)")

In [ ]:
# ── CELL 14 — Grammar scoring (uses the SAME ISP model) ──────────────────────
# Mirrors the grammar-score-checker notebook, but routed through the currently
# loaded ISP model via `complete_grammar` (chat-format inference, greedy, 5 tokens).
language_tool = language_tool_python.LanguageTool("en-US")
print("✅ LanguageTool ready (fallback)")


def complete_grammar(messages, max_tokens=5):
    """Deterministic short completion for grammar scoring (same model)."""
    text = build_chat_text(messages)
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    return tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n").strip()


def language_tool_fallback(text):
    matches = language_tool.check(text)
    score = 100
    for match in matches:
        if match.ruleId.startswith("MORFOLOGIK_") or match.ruleId == "UPPERCASE_SENTENCE_START":
            score -= 5
        elif "AGREEMENT" in match.ruleId:
            score -= 15
        elif "GRAMMAR" in match.ruleId or "SENTENCE" in match.ruleId:
            score -= 20
        else:
            score -= 10
    return max(0, score)


def get_llm_grammar_score(text):
    if not text or not text.strip():
        return 0
    system_prompt = (
        "You are a strict grammar evaluator. Score grammar from 0 to 100. "
        "100 = perfect grammar. 0 = very poor grammar. Return only a number, no text."
    )
    user_prompt = (
        "Evaluate the grammar of this text and return a score from 0 to 100.\n"
        'Text:\n"""\n' + text + '\n"""'
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_prompt},
    ]
    try:
        raw_output = complete_grammar(messages, max_tokens=5)
        match = re.match(r'^\[?(\d+)\]?$', raw_output.strip())
        if match:
            score = int(match.group(1))
        else:
            numbers = re.findall(r'\d+', raw_output)
            score = int(numbers[0]) if numbers else None
            if score is None:
                raise ValueError("No valid number found")
        return score if 0 <= score <= 100 else language_tool_fallback(text)
    except (ValueError, TypeError):
        return language_tool_fallback(text)
    except RuntimeError as e:
        if "CUDA out of memory" in str(e):
            torch.cuda.empty_cache(); gc.collect()
            return language_tool_fallback(text)
        raise


print("✅ Grammar scoring ready (same model + LanguageTool fallback)")

In [ ]:
# ── CELL 15 — Prompt improvement helpers (critique & patch — verbatim) ───────
STOP_TRIGGERS = (
    "note:", "explanation:", "this prompt", "i improved",
    "the above", "rationale:", "in this version",
    "here is", "here's", "the following", "this revised",
)


def prepare_prompt_for_display(prompt_template):
    if not USE_WORD_COUNT:
        return prompt_template
    return prompt_template.replace(PLACEHOLDER_WORD, str(DISPLAY_WORD_COUNT))


def restore_placeholder(improved_prompt):
    if not USE_WORD_COUNT:
        return improved_prompt   # never re-insert a word target
    for pat in [r'approximately\s+\d+[-–]?\d*\s+words?',
                r'about\s+\d+[-–]?\d*\s+words?',
                r'around\s+\d+[-–]?\d*\s+words?']:
        improved_prompt = re.sub(pat, f'approximately {PLACEHOLDER_WORD} words',
                                 improved_prompt, flags=re.IGNORECASE)
    improved_prompt = re.sub(r'\b\d+\s+words?\b', f'{PLACEHOLDER_WORD} words',
                             improved_prompt, flags=re.IGNORECASE)
    return improved_prompt


def critique_prompt(prompt_template, rouge1, rouge2, rougeL, bert_f1):
    display_prompt = prepare_prompt_for_display(prompt_template)
    instruction = (
        "A language model used the following prompt to summarize clinical "
        "case reports and received these evaluation scores (0-1, higher is better):\n\n"
        f"  ROUGE-1   : {rouge1:.4f}\n"
        f"  ROUGE-2   : {rouge2:.4f}\n"
        f"  ROUGE-L   : {rougeL:.4f}\n"
        f"  BERTScore : {bert_f1:.4f}\n\n"
        "━━━ PROMPT ━━━\n"
        f"{display_prompt}\n\n"
        "Identify exactly 3 specific weaknesses in this prompt that may be "
        "causing low scores. Consider: clarity, completeness of required "
        "clinical content, output format, and generalizability.\n\n"
        "Write ONLY 3 bullet points. One weakness per bullet.\n"
        "Do NOT rewrite the prompt. No other text.\n"
    )
    raw      = generate_model_response(instruction, MAX_CRITIQUE_TOKENS, temperature=0.5)
    critique = ("• " + raw).strip()
    critique = re.sub(r"^(weaknesses\s*[:\-]*\s*)", "", critique, flags=re.IGNORECASE).strip()
    return critique


def patch_prompt(prompt_template, critique):
    display_prompt = prepare_prompt_for_display(prompt_template)
    rule4 = (f"4. Keep the word count target as 'approximately {DISPLAY_WORD_COUNT} words'.\n"
             if USE_WORD_COUNT else
             "4. Do NOT mention any specific word count or length target.\n")
    instruction = (
        "Below is a clinical summarization prompt and a critique of its weaknesses.\n\n"
        "━━━ ORIGINAL PROMPT ━━━\n"
        f"{display_prompt}\n\n"
        "━━━ WEAKNESSES ━━━\n"
        f"{critique}\n\n"
        "Rewrite the prompt fixing ONLY the identified weaknesses. Keeping the clinical "
        "summarisation thing in mind generate the Prompt in that particular manner and it "
        "should well explain on what task it is going to perform.\n"
        "Rules:\n"
        "1. Keep the overall structure and intent of the original prompt same.\n"
        "2. Make only minimal targeted changes.\n"
        "3. Must be general — valid for ANY clinical case.\n"
        f"{rule4}"
        "5. Cover: demographics, complaint, investigations, diagnosis, treatment, outcome.\n"
        "6. Numbered items WITHOUT blank lines between them.\n"
        "7. Return ONLY the improved prompt. No title. No explanation.\n"
    )
    raw = generate_model_response(instruction, MAX_PATCH_TOKENS, temperature=0.6)

    improved = re.sub(r"^(improved prompt\s*[:\-]*\s*)", "", raw, flags=re.IGNORECASE).strip()
    improved = improved.strip().strip(chr(34)*3).strip(chr(39)*3).strip()

    clean_lines = []
    for line in improved.split("\n"):
        if any(line.strip().lower().startswith(t) for t in STOP_TRIGGERS):
            break
        clean_lines.append(line)
    improved = "\n".join(clean_lines).strip()

    if len(improved) < 40:
        print("  ⚠️  Patch too short — keeping current prompt.")
        return prompt_template

    improved = restore_placeholder(improved)
    if USE_WORD_COUNT and PLACEHOLDER_WORD not in improved:
        improved = improved.replace(
            f"approximately {DISPLAY_WORD_COUNT} words",
            f"approximately {PLACEHOLDER_WORD} words")
        if PLACEHOLDER_WORD not in improved:
            improved = improved.rstrip('.') + f" Write approximately {PLACEHOLDER_WORD} words."

    print(f"  ✅ Improved prompt ready ({count_tokens(improved)} tokens)")
    if USE_WORD_COUNT:
        print(f"  ✅ Placeholder : {'✅ present' if PLACEHOLDER_WORD in improved else '❌ MISSING'}")
    return improved


print("✅ Prompt improvement helpers ready")

In [ ]:
# ── CELL 16 — Scoring helper (entailment + grammar + summarisation score) ────
rouge_metric = hf_evaluate.load("rouge")


def score_prompt_on_dataset(prompt_template, dataset, label=""):
    safe_label = label.replace(" ", "_").replace("/", "_")
    run_ckpt   = os.path.join(OUTPUT_DIR, f"{MODEL_TAG}_run_ckpt_{safe_label}.json")

    completed_rows = []
    start_idx      = 0
    if os.path.exists(run_ckpt):
        with open(run_ckpt) as f:
            rc = json.load(f)
        completed_rows = rc["completed_rows"]
        start_idx      = len(completed_rows)
        print(f"  ✅ Run checkpoint found: resuming from case {start_idx + 1}/{len(dataset)}")
    else:
        print(f"  Starting fresh — no run checkpoint for [{label}]")

    print(f"\n  Generating summaries + entailment for {len(dataset)} cases ...")

    new_rows = []
    for i in range(start_idx, len(dataset)):
        example = dataset[i]

        # 1. Generate summary (fixed 150 tokens)
        generated, max_new, min_new, tw, ref_len = generate_summary(
            prompt_template, example["case"], example["summary"])
        gen_len = count_tokens(generated)

        # 2. Derived flags
        within_budget = bool(gen_len <= max_new)
        above_minimum = bool(gen_len >= min_new)
        is_complete   = bool(generated.strip() and generated.strip()[-1] in ('.', '!', '?'))
        ratio         = round(gen_len / ref_len, 2) if ref_len > 0 else 0.0

        # 3. Entailment
        ent_result = compute_entailment(example["case"], example["summary"], generated)

        # 4. Grammar score (same model)
        grammar = get_llm_grammar_score(generated)

        if (i + 1) % 25 == 0 or i == len(dataset) - 1:
            print(f"  [{i+1:>4}/{len(dataset)}] {example['filename']:<32} "
                  f"gen={gen_len:>3} ratio={ratio:>4} gram={grammar:>3} both_ent={ent_result['both_entailed']}")

        new_rows.append({
            "filename":                    example["filename"],
            "prompt_used":                 prompt_template,
            "reference_summary":           example["summary"],
            "generated_summary":           generated,
            "entailment_prob_reference":   ent_result["entailment_prob_reference"],
            "neutral_prob_reference":      ent_result["neutral_prob_reference"],
            "contradiction_prob_reference":ent_result["contradiction_prob_reference"],
            "reference_entailed":          ent_result["reference_entailed"],
            "entailment_prob_generated":   ent_result["entailment_prob_generated"],
            "neutral_prob_generated":      ent_result["neutral_prob_generated"],
            "contradiction_prob_generated":ent_result["contradiction_prob_generated"],
            "generated_entailed":          ent_result["generated_entailed"],
            "both_entailed":               ent_result["both_entailed"],
            "rouge1": None, "rouge2": None, "rougeL": None,
            "bert_precision": None, "bert_recall": None, "bert_f1": None,
            "grammar_score":   grammar,
            "gen_token_count": gen_len,
            "ref_token_count": ref_len,
            "within_budget":   within_budget,
            "above_minimum":   above_minimum,
            "is_complete_sentence": is_complete,
        })

        if (i + 1) % CHECKPOINT_EVERY == 0 or i == len(dataset) - 1:
            with open(run_ckpt, "w") as f:
                json.dump({"completed_rows": completed_rows + new_rows}, f)

    all_rows = completed_rows + new_rows
    preds    = [r["generated_summary"] for r in all_rows]
    refs     = [r["reference_summary"]  for r in all_rows]

    # ROUGE
    print(f"\n  Computing ROUGE ...")
    for idx, (pred, ref) in enumerate(zip(preds, refs)):
        if all_rows[idx]["rouge1"] is not None:
            continue
        s = rouge_metric.compute(predictions=[pred], references=[ref], use_stemmer=True)
        all_rows[idx]["rouge1"] = round(s["rouge1"], 4)
        all_rows[idx]["rouge2"] = round(s["rouge2"], 4)
        all_rows[idx]["rougeL"] = round(s["rougeL"], 4)
    print(f"  ✅ ROUGE done")

    # BERTScore
    print(f"  Computing BERTScore ...")
    for b_start in range(0, len(all_rows), 50):
        b_end = min(b_start + 50, len(all_rows))
        if all_rows[b_start]["bert_f1"] is not None:
            continue
        P_b, R_b, F1_b = bert_score_fn(
            preds[b_start:b_end], refs[b_start:b_end],
            model_type=BERT_MODEL, lang="en", verbose=False)
        for j in range(b_end - b_start):
            all_rows[b_start+j]["bert_precision"] = round(P_b[j].item(), 4)
            all_rows[b_start+j]["bert_recall"]    = round(R_b[j].item(), 4)
            all_rows[b_start+j]["bert_f1"]        = round(F1_b[j].item(), 4)
    print(f"  ✅ BERTScore done")

    # Per-row summarisation score = 0.6*rougeL + 0.4*bert_f1 (single case)
    for r in all_rows:
        r["summarisation_score"] = round(W_ROUGEL * r["rougeL"] + W_BERT * r["bert_f1"], 4)

    df  = pd.DataFrame(all_rows)
    avg_rougeL  = round(df["rougeL"].mean(), 4)
    avg_bert_f1 = round(df["bert_f1"].mean(), 4)
    avg_grammar = round(df["grammar_score"].mean(), 2)
    summ_score  = round(W_ROUGEL * avg_rougeL + W_BERT * avg_bert_f1, 4)

    agg = {
        "rouge1":  round(df["rouge1"].mean(), 4),
        "rouge2":  round(df["rouge2"].mean(), 4),
        "rougeL":  avg_rougeL,
        "bert_p":  round(df["bert_precision"].mean(), 4),
        "bert_r":  round(df["bert_recall"].mean(), 4),
        "bert_f1": avg_bert_f1,
        "grammar":              avg_grammar,
        "summarisation_score":  summ_score,
    }

    both_ent_pct = round(df["both_entailed"].mean() * 100, 1)
    ref_ent_pct  = round(df["reference_entailed"].mean() * 100, 1)
    gen_ent_pct  = round(df["generated_entailed"].mean() * 100, 1)
    complete_pct  = round(df["is_complete_sentence"].mean() * 100, 1)

    print(f"\n  [{label}] Scores ({MODEL_TAG}):")
    print(f"  Avg ROUGE-L={agg['rougeL']}  Avg BERT-F1={agg['bert_f1']}  Avg Grammar={agg['grammar']}")
    print(f"  ★ Summarisation score (0.6·RL + 0.4·BERT) = {agg['summarisation_score']}")
    print(f"  Entailment → ref:{ref_ent_pct}%  gen:{gen_ent_pct}%  both:{both_ent_pct}%")
    print(f"  Complete sentences:{complete_pct}%")

    if os.path.exists(run_ckpt):
        os.remove(run_ckpt)

    csv_cols = [
        "filename", "reference_summary", "generated_summary",
        "entailment_prob_reference", "neutral_prob_reference", "contradiction_prob_reference", "reference_entailed",
        "entailment_prob_generated",  "neutral_prob_generated",  "contradiction_prob_generated",  "generated_entailed",
        "both_entailed",
        "rouge1", "rouge2", "rougeL",
        "bert_precision", "bert_recall", "bert_f1",
        "grammar_score", "summarisation_score",
        "within_budget", "is_complete_sentence",
    ]
    all_rows_clean = [{k: r[k] for k in csv_cols} for r in all_rows]
    return agg, all_rows, all_rows_clean


print("✅ Scoring helper ready — entailment + grammar + summarisation score")

In [ ]:
# ── CELL 17 — Initial prompt & iteration checkpoint restore ──────────────────
_length_line = (
    "in approximately {target_words} words.\n" if USE_WORD_COUNT else "in clear, continuous prose.\n"
)
initial_prompt = (
    "Write a concise clinical summary of the following case report "
    + _length_line +
    "Cover all of these points without blank lines between them:\n"
    "1. Patient demographics (age, sex)\n"
    "2. Presenting complaint and relevant medical history\n"
    "3. Key clinical findings and investigations\n"
    "4. Diagnosis\n"
    "5. Treatment plan\n"
    "6. Clinical outcome\n"
    "Use clear, professional medical language.\n"
    "Do NOT use bullet points or headers — write in continuous prose.\n"
    "Complete every sentence fully before stopping."
)

print("Initial prompt:")
print(prepare_prompt_for_display(initial_prompt))
if USE_WORD_COUNT:
    print(f"\nPlaceholder check: {'✅ present' if PLACEHOLDER_WORD in initial_prompt else '❌ MISSING'}")
else:
    print(f"\nWord-count instruction: OFF (length controlled by {FIXED_TOKENS}-token cap)")

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    current_prompt       = ckpt["current_prompt"]
    best_prompt          = ckpt["best_prompt"]
    best_summ_score      = ckpt["best_summ_score"]
    best_rougeL          = ckpt["best_rougeL"]
    best_bert_f1         = ckpt["best_bert_f1"]
    prompt_history       = ckpt["prompt_history"]
    completed_iterations = ckpt["completed_iterations"]
    all_iter_rows        = ckpt.get("all_iter_rows", [])
    start_iteration      = ckpt["current_iteration"]
    print(f"\n✅ Resuming from iteration {start_iteration}")
else:
    current_prompt       = initial_prompt
    best_prompt          = initial_prompt
    best_summ_score      = -1.0
    best_rougeL          = 0.0
    best_bert_f1         = 0.0
    prompt_history       = []
    completed_iterations = []
    all_iter_rows        = []
    start_iteration      = 1
    print(f"\n✅ Starting fresh")

In [ ]:
# ── CELL 18 — ISP loop (best prompt tracked by SUMMARISATION SCORE) ──────────
for iteration in range(start_iteration, ITERATIONS + 1):

    print(f"\n{'╔' + '═'*68 + '╗'}")
    print(f"║  ITERATION {iteration} / {ITERATIONS} — {MODEL_TAG} — {SEARCH_CASES} cases")
    print(f"╚{'═'*68}╝")

    if USE_WORD_COUNT and PLACEHOLDER_WORD not in current_prompt:
        current_prompt = restore_placeholder(current_prompt)
        if PLACEHOLDER_WORD not in current_prompt:
            current_prompt += f" Write approximately {PLACEHOLDER_WORD} words."

    print(f"\n  Current prompt (first 150 chars):")
    print(f"  {prepare_prompt_for_display(current_prompt)[:150]}...")

    agg, iter_rows, _ = score_prompt_on_dataset(
        current_prompt, search_dataset, label=f"Iteration {iteration}")

    for r in iter_rows:
        r["iteration"] = iteration
    all_iter_rows.extend(iter_rows)

    # ── Prompt grammar score (once per iteration; prompt is constant) ─────────
    # Scored on the display version so {target_words} (if any) is filled first.
    prompt_grammar = get_llm_grammar_score(prepare_prompt_for_display(current_prompt))
    print(f"\n  📝 Prompt grammar score : {prompt_grammar} / 100")

    # ── Best prompt selection by summarisation score ─────────────────────────
    this_summ = agg["summarisation_score"]
    if this_summ > best_summ_score:
        best_prompt     = current_prompt
        best_summ_score = this_summ
        best_rougeL     = agg["rougeL"]
        best_bert_f1    = agg["bert_f1"]
        print(f"\n  🏆 NEW BEST PROMPT (iteration {iteration})")
        print(f"     Summarisation score={best_summ_score}  ROUGE-L={best_rougeL}  BERT-F1={best_bert_f1}")
    else:
        print(f"\n  ℹ️  No improvement. Best summ score so far = {best_summ_score}")

    if iteration < ITERATIONS:
        print(f"\n  Step 1 — Critiquing prompt ...")
        critique = critique_prompt(current_prompt, agg["rouge1"], agg["rouge2"], agg["rougeL"], agg["bert_f1"])
        print(f"  Critique:\n  {critique[:200]}")

        print(f"\n  Step 2 — Patching prompt ...")
        new_prompt = patch_prompt(current_prompt, critique)
        print(f"\n  New prompt (first 200 chars):")
        print(f"  {prepare_prompt_for_display(new_prompt)[:200]}")

        prompt_history.append({
            "iteration": iteration, "prompt_used": current_prompt,
            "rouge1": agg["rouge1"], "rouge2": agg["rouge2"], "rougeL": agg["rougeL"],
            "bert_f1": agg["bert_f1"], "grammar": agg["grammar"],
            "prompt_grammar": prompt_grammar,
            "summarisation_score": this_summ,
            "critique": critique, "new_prompt": new_prompt,
        })
        current_prompt = new_prompt
    else:
        prompt_history.append({
            "iteration": iteration, "prompt_used": current_prompt,
            "rouge1": agg["rouge1"], "rouge2": agg["rouge2"], "rougeL": agg["rougeL"],
            "bert_f1": agg["bert_f1"], "grammar": agg["grammar"],
            "prompt_grammar": prompt_grammar,
            "summarisation_score": this_summ,
            "critique": None, "new_prompt": None,
        })

    completed_iterations.append(iteration)

    ckpt_state = {
        "current_prompt": current_prompt, "best_prompt": best_prompt,
        "best_summ_score": best_summ_score, "best_rougeL": best_rougeL, "best_bert_f1": best_bert_f1,
        "prompt_history": prompt_history, "completed_iterations": completed_iterations,
        "all_iter_rows": all_iter_rows, "current_iteration": iteration + 1,
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(ckpt_state, f, indent=2)
    print(f"\n  💾 Iteration checkpoint saved (iteration {iteration} complete)")

In [ ]:
# ── CELL 19 — Save iteration results, prompt history & BEST PROMPT ───────────
df_iter = pd.DataFrame(all_iter_rows)
df_iter.to_csv(ITER_RESULTS_CSV, index=False)
print(f"✅ Iteration results saved → {ITER_RESULTS_CSV}")

# Identify best-performing prompt explicitly (by summarisation score)
best_entry = max(prompt_history, key=lambda h: h["summarisation_score"])

with open(PROMPT_HIST_JSON, "w") as f:
    json.dump({
        "model": MODEL_TAG, "model_repo": MODEL_NAME,
        "best_prompt": best_prompt,
        "best_summ_score": best_summ_score,
        "best_rougeL": best_rougeL, "best_bert_f1": best_bert_f1,
        "best_iteration": best_entry["iteration"],
        "history": prompt_history,
    }, f, indent=2)
print(f"✅ Prompt history saved → {PROMPT_HIST_JSON}")

print(f"\n{'═'*78}")
print(f"  CROSS-ITERATION COMPARISON — {MODEL_TAG}")
print(f"{'═'*78}")
print(f"  {'Iter':<6} {'ROUGE-L':>9} {'BERT-F1':>9} {'SumGram':>9} {'PmtGram':>9} {'SummScore':>11}")
print(f"  {'─'*65}")
for h in prompt_history:
    marker = "  ← BEST" if h["prompt_used"] == best_prompt else ""
    print(f"  {h['iteration']:<6} {h['rougeL']:>9.4f} {h['bert_f1']:>9.4f} "
          f"{h['grammar']:>9.2f} {h.get('prompt_grammar', 0):>9.2f} "
          f"{h['summarisation_score']:>11.4f}{marker}")

print(f"\n  🏆 BEST-PERFORMING PROMPT (iteration {best_entry['iteration']}, "
      f"summ score={best_summ_score}):\n")
print(prepare_prompt_for_display(best_prompt))

In [ ]:
# ── CELL 20 — Final evaluation of BEST prompt on ALL cases ───────────────────
print(f"\n{'╔' + '═'*68 + '╗'}")
print(f"║  FINAL EVALUATION — Best prompt on ALL {len(short_dataset)} cases — {MODEL_TAG}")
print(f"╚{'═'*68}╝\n")
print(f"Best prompt:\n{prepare_prompt_for_display(best_prompt)}\n")

# ── Decide: reuse existing rows or run a fresh final pass ─────────────────────
# Reuse is valid only if the search set covered the FULL dataset (so the best
# prompt was already scored on every case) and the flag is on.
_search_is_full = (len(search_dataset) == len(short_dataset))
_best_iter      = best_entry["iteration"]
_reuse_rows     = [r for r in all_iter_rows if r.get("iteration") == _best_iter]

if SKIP_REDUNDANT_FINAL and _search_is_full and len(_reuse_rows) == len(short_dataset):
    print(f"⏩ Skipping redundant 4th run — reusing iteration {_best_iter} rows "
          f"(best prompt already scored on all {len(short_dataset)} cases).")
    csv_cols = [
        "filename", "reference_summary", "generated_summary",
        "entailment_prob_reference", "neutral_prob_reference", "contradiction_prob_reference", "reference_entailed",
        "entailment_prob_generated",  "neutral_prob_generated",  "contradiction_prob_generated",  "generated_entailed",
        "both_entailed",
        "rouge1", "rouge2", "rougeL",
        "bert_precision", "bert_recall", "bert_f1",
        "grammar_score", "summarisation_score",
        "within_budget", "is_complete_sentence",
    ]
    final_rows_clean = [{k: r[k] for k in csv_cols} for r in _reuse_rows]
    final_rows       = _reuse_rows
else:
    if SKIP_REDUNDANT_FINAL and not _search_is_full:
        print(f"ℹ️  Search set ({len(search_dataset)}) < full set ({len(short_dataset)}) "
              f"— running fresh final pass on all cases.")
    final_agg, final_rows, final_rows_clean = score_prompt_on_dataset(
        best_prompt, short_dataset, label="FINAL")

df_final = pd.DataFrame(final_rows_clean)
df_final.to_csv(FINAL_CSV, index=False)

if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

# ── Aggregate metrics requested ──────────────────────────────────────────────
avg_rougeL  = round(df_final["rougeL"].mean(), 4)
avg_bert_f1 = round(df_final["bert_f1"].mean(), 4)
avg_grammar = round(df_final["grammar_score"].mean(), 2)
summ_score  = round(W_ROUGEL * avg_rougeL + W_BERT * avg_bert_f1, 4)

metrics = {
    "model":               MODEL_TAG,
    "model_repo":          MODEL_NAME,
    "n_cases":             int(len(df_final)),
    "fixed_tokens":        FIXED_TOKENS,
    "avg_rougeL":          avg_rougeL,
    "avg_bert_f1":         avg_bert_f1,
    "avg_summary_grammar_score": avg_grammar,   # mean grammar of generated summaries
    "best_prompt_grammar_score": best_entry.get("prompt_grammar"),  # grammar of the winning prompt
    "summarisation_score": summ_score,   # 0.6·avg ROUGE-L + 0.4·avg BERT-F1
    "best_prompt":         best_prompt,
    "best_iteration":      best_entry["iteration"],
}
with open(METRICS_JSON, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\n✅ Final results saved → {FINAL_CSV}  ({len(df_final)} rows)")
print(f"✅ Metrics summary saved → {METRICS_JSON}")

print(f"\n{'═'*65}")
print(f"  ★ FINAL AGGREGATE METRICS — {MODEL_TAG} ({len(df_final)} cases)")
print(f"{'═'*65}")
print(f"  Average ROUGE-L        : {avg_rougeL}")
print(f"  Average BERTScore (F1) : {avg_bert_f1}")
print(f"  Avg summary grammar    : {avg_grammar} / 100")
print(f"  Best prompt grammar    : {best_entry.get('prompt_grammar')} / 100")
print(f"  Summarisation score    : {summ_score}   (= {W_ROUGEL}·ROUGE-L + {W_BERT}·BERT-F1)")
print(f"{'═'*65}")
print(f"\n  Entailment Summary:")
print(f"  Reference entailed : {df_final['reference_entailed'].mean()*100:.1f}%")
print(f"  Generated entailed : {df_final['generated_entailed'].mean()*100:.1f}%")
print(f"  Both entailed      : {df_final['both_entailed'].mean()*100:.1f}%")
print(f"\n  Token Quality:")
print(f"  Complete sentences   : {df_final['is_complete_sentence'].mean()*100:.1f}%")

In [ ]:
# ── CELL 21 — Graphs (top/bottom cases) ──────────────────────────────────────
def plot_bar_graph(data_df, score_col, title, ylabel, color, save_path):
    fig, ax = plt.subplots(figsize=(13, 6))
    filenames   = data_df["filename"].tolist()
    scores      = data_df[score_col].tolist()
    short_names = [f.replace("multiclinsum_gs_en_", "case_") for f in filenames]
    bars = ax.bar(range(len(short_names)), scores, color=color, edgecolor="black", linewidth=0.7, alpha=0.85)
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{score:.4f}", ha="center", va="bottom", fontsize=9, fontweight="bold")
    ax.set_xticks(range(len(short_names)))
    ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=9)
    ax.set_ylim(max(0, min(scores)-0.05), min(1.0, max(scores)+0.08))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.set_ylabel(ylabel, fontsize=12); ax.set_xlabel("Case", fontsize=12)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=15)
    ax.grid(axis="y", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.show()
    print(f"  ✅ Saved → {save_path}")

_T = MODEL_DISPLAY
plot_bar_graph(df_final.nlargest(10, "rougeL")[["filename","rougeL"]].reset_index(drop=True),
    "rougeL", f"Top 10 — Highest ROUGE-L\n({_T} ISP)", "ROUGE-L", "#2ecc71", GRAPH_HIGH_ROUGE)
plot_bar_graph(df_final.nsmallest(10, "rougeL")[["filename","rougeL"]].reset_index(drop=True),
    "rougeL", f"Top 10 — Lowest ROUGE-L\n({_T} ISP)", "ROUGE-L", "#e74c3c", GRAPH_LOW_ROUGE)
plot_bar_graph(df_final.nlargest(10, "bert_f1")[["filename","bert_f1"]].reset_index(drop=True),
    "bert_f1", f"Top 10 — Highest BERTScore F1\n({_T} ISP)", "BERTScore F1", "#3498db", GRAPH_HIGH_BERT)
plot_bar_graph(df_final.nsmallest(10, "bert_f1")[["filename","bert_f1"]].reset_index(drop=True),
    "bert_f1", f"Top 10 — Lowest BERTScore F1\n({_T} ISP)", "BERTScore F1", "#e67e22", GRAPH_LOW_BERT)

In [ ]:
# ── CELL 22 — Iteration trend graphs ─────────────────────────────────────────
iterations_list = [h["iteration"] for h in prompt_history]
rouge1_scores   = [h["rouge1"]    for h in prompt_history]
rouge2_scores   = [h["rouge2"]    for h in prompt_history]
rougeL_scores   = [h["rougeL"]    for h in prompt_history]
bert_f1_scores  = [h["bert_f1"]   for h in prompt_history]
x      = np.arange(len(iterations_list))
labels = [f"Iteration {i}" for i in iterations_list]

# ROUGE grouped bar
fig, ax = plt.subplots(figsize=(10, 6))
bw = 0.25
ax.bar(x - bw, rouge1_scores, bw, label="ROUGE-1", color="#3498db", edgecolor="black", linewidth=0.7, alpha=0.88)
ax.bar(x,      rouge2_scores, bw, label="ROUGE-2", color="#2ecc71", edgecolor="black", linewidth=0.7, alpha=0.88)
ax.bar(x + bw, rougeL_scores, bw, label="ROUGE-L", color="#e74c3c", edgecolor="black", linewidth=0.7, alpha=0.88)
for xs, ys in [(x-bw,rouge1_scores),(x,rouge2_scores),(x+bw,rougeL_scores)]:
    for xi, yi in zip(xs, ys):
        ax.text(xi, yi + 0.003, f"{yi:.4f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("Score", fontsize=12); ax.set_xlabel("ISP Iteration", fontsize=12)
ax.set_title(f"ROUGE Across ISP Iterations\n({MODEL_DISPLAY} — {SEARCH_CASES} cases)", fontsize=13, fontweight="bold", pad=15)
_all = rouge1_scores + rouge2_scores + rougeL_scores
ax.set_ylim(max(0, min(_all)-0.05), min(1.0, max(_all)+0.08))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
ax.legend(fontsize=10, loc="lower right"); ax.grid(axis="y", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(GRAPH_ROUGE_COMP, dpi=150, bbox_inches="tight"); plt.show()
print(f"✅ Saved → {GRAPH_ROUGE_COMP}")

# BERTScore F1 line
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(x, bert_f1_scores, color="#8e44ad", linewidth=2.5, marker="o", markersize=10,
        markerfacecolor="white", markeredgecolor="#8e44ad", markeredgewidth=2.5, label="BERTScore F1")
for xi, val in zip(x, bert_f1_scores):
    ax.text(xi, val + 0.001, f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold", color="#8e44ad")
ax.fill_between(x, bert_f1_scores, alpha=0.12, color="#8e44ad")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("BERTScore F1", fontsize=12); ax.set_xlabel("ISP Iteration", fontsize=12)
ax.set_title(f"BERTScore F1 Across ISP Iterations\n({MODEL_DISPLAY} — {SEARCH_CASES} cases)", fontsize=13, fontweight="bold", pad=15)
ax.set_ylim(max(0.80, min(bert_f1_scores)-0.01), min(1.0, max(bert_f1_scores)+0.01))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
ax.legend(fontsize=10, loc="lower right"); ax.grid(axis="y", linestyle="--", alpha=0.5); ax.set_axisbelow(True)
plt.tight_layout(); plt.savefig(GRAPH_BERT_COMP, dpi=150, bbox_inches="tight"); plt.show()
print(f"✅ Saved → {GRAPH_BERT_COMP}")

# ── Score comparison table rendered as an image ──────────────────────────────
fig, ax = plt.subplots(figsize=(9.5, 0.7 + 0.5*len(prompt_history)))
ax.axis("off")
header = ["Iter", "R-1", "R-2", "R-L", "BERT-F1", "SumGram", "PmtGram", "SummScore", ""]
best_i = max(range(len(prompt_history)), key=lambda k: prompt_history[k]["summarisation_score"])
rows = []
for k, h in enumerate(prompt_history):
    rows.append([
        f"Iteration {h['iteration']}",
        f"{h['rouge1']:.4f}", f"{h['rouge2']:.4f}", f"{h['rougeL']:.4f}",
        f"{h['bert_f1']:.4f}", f"{h['grammar']:.2f}", f"{h.get('prompt_grammar', 0):.2f}",
        f"{h['summarisation_score']:.4f}",
        "← BEST" if k == best_i else "",
    ])
tbl = ax.table(cellText=rows, colLabels=header, cellLoc="left", loc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 1.6)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("white")
    if r == 0:
        cell.set_text_props(fontweight="bold"); cell.set_facecolor("#f0f0f0")
    elif r-1 == best_i:
        cell.set_facecolor("#eafaf0")
ax.set_title(f"SCORE COMPARISON ACROSS {len(prompt_history)} ITERATIONS\n({MODEL_DISPLAY})",
             fontsize=12, fontweight="bold", pad=12, family="monospace")
plt.tight_layout(); plt.savefig(GRAPH_SCORE_TBL, dpi=150, bbox_inches="tight"); plt.show()
print(f"✅ Saved → {GRAPH_SCORE_TBL}")

print(f"\n✅ All outputs saved to {OUTPUT_DIR}  (prefix: {MODEL_TAG}_)")
print(f"   📄 {os.path.basename(FINAL_CSV)}")
print(f"   📄 {os.path.basename(ITER_RESULTS_CSV)}")
print(f"   📋 {os.path.basename(PROMPT_HIST_JSON)}")
print(f"   📊 {os.path.basename(METRICS_JSON)}")
print(f"\n🎉 Generalised ISP complete for {MODEL_TAG}!")